- LLM이 Text2Cypher를 수행하는데 필요한 개념적인 정보들을 그래프에 추가하기 위한 과정
- 일단은 수작업으로 생성하고, 추후에 자동적으로 생성할 수 있는 방법이 있는지 고민

In [ ]:
from neo4j import GraphDatabase

# Neo4j 연결 정보
uri = "bolt://neo4j-gds-apoc-n10s:7687"
username = "neo4j"
password = "neo4jpassword"

In [ ]:
driver = GraphDatabase.driver(uri, auth=(username, password))

In [ ]:
concept_silver = """
MATCH (p:`요금제`)-[:`가입조건`]->(a:`가입조건`)
WHERE a.`가입가능최소나이` >= 65
MERGE (c:`개념` {`키워드`: '실버요금제'})
ON CREATE SET
    c.`설명` = 'Plans for customers aged 65 and older',
    c.CYPHER_TEMPLATE = 'MATCH (p:`요금제`)-[:`가입조건`]->(a:`가입조건`) WHERE a.`가입가능최소나이` >= 65 RETURN p'
MERGE (p)-[:`연관`]->(c)
"""

concept_senior = """
MATCH (p:`요금제`)-[:`가입조건`]->(a:`가입조건`)
WHERE a.`가입가능최소나이` >= 65
MERGE (c:`개념` {`키워드`: '시니어요금제'})
ON CREATE SET
    c.설명 = 'Plans for customers aged 65 and older',
    c.CYPHER_TEMPLATE = 'MATCH (p:`요금제`)-[:`가입조건`]->(a:`가입조건`) WHERE a.`가입가능최소나이` >= 65 RETURN p'
MERGE (p)-[:`연관`]->(c)
"""

concept_payg = """
MATCH (p:`요금제`)-[:`제공`]->(d:`데이터용량`)
WHERE d.`기본제공데이터용량` = 0
MERGE (c:`개념` {`키워드`: '종량제요금제'})
ON CREATE SET
    c.설명 = 'Pay-as-you-go plans',
    c.CYPHER_TEMPLATE = 'MATCH (p:`요금제`)-[:`제공`]->(d:`데이터용량`) WHERE d.`기본제공데이터용량` = 0 RETURN p'
MERGE (p)-[:`연관`]->(c)
"""

concept_music_streaming = """
MERGE (c:개념 {키워드: '음악듣기요금제'})
  ON CREATE SET
    c.설명     = 'Plans that offer music streaming services',
    c.CYPHER_TEMPLATE = 'MATCH (p:요금제)-[:속함]->(:요금제그룹 {그룹명: "유튜브 프리미엄 요금제"}) RETURN p UNION MATCH (p:요금제) WHERE ANY(k IN p.마케팅키워드 WHERE k CONTAINS "FLO") RETURN p'
WITH c
CALL () {
  MATCH (p:요금제)-[:속함]->(:요금제그룹 {그룹명: '유튜브 프리미엄 요금제'})
  RETURN p
  UNION
  MATCH (p:요금제)
  WHERE ANY(k IN p.마케팅키워드 WHERE k CONTAINS 'FLO')
  RETURN p
}
WITH c, p
MERGE (p)-[:연관]->(c)
"""

concept_soldier = """
MERGE (c:개념 {키워드: '장교'})
  ON CREATE SET
    c.설명 = 'Military plans are not available to professional soldiers'
"""

concept_discount = """
MERGE (c:개념 {키워드: '요금 할인'})
  ON CREATE SET
    c.설명 = 'Price discounts means that the discount is applied to the monthly fee of the plan'
"""

In [ ]:
concept_cypher_list = [concept_silver, concept_senior, concept_payg, concept_music_streaming, concept_soldier, concept_discount]

In [ ]:
# 모든 개념 노드 초기화
with driver.session() as session:
    session.run("MATCH (n:`개념`) DETACH DELETE n")

In [ ]:
with driver.session() as session:
    for cypher_query in concept_cypher_list:
        session.run(cypher_query)

In [ ]:
driver.close()